In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

# Set Chinese font and style
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

In [ ]:
# 1. Load data
df = pd.read_csv('csv/df_k.csv')

# 2. Only take data with kl < 20
df_filtered = df[df['k(W/mK)'] < 10].copy()
print(f"Original data count: {len(df)}")
print(f"Filtered data count (k < 10): {len(df_filtered)}")

# 3. Prepare data
X = df_filtered.drop(columns=['k(W/mK)', 'formula', 'composition'])
y = df_filtered['k(W/mK)']

print(f"Feature count: {X.shape[1]}")
print(f"Target variable range: [{y.min():.2f}, {y.max():.2f}]")
print(f"Target variable mean: {y.mean():.2f}")

# 4. Missing value handling
print(f'Feature missing values count: {X.isnull().sum().sum()}')
print(f'Target variable missing values count: {y.isnull().sum()}')

if X.isnull().sum().sum() > 0:
    X = X.fillna(X.mean())
if y.isnull().sum() > 0:
    y = y.fillna(y.mean())

In [ ]:
# 5. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set size: {X_train.shape}')
print(f'Testing set size: {X_test.shape}')

# 6. Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 7. Create XGBoost regression model
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

# 8. Train model
xgb_model.fit(X_train_scaled, y_train)

In [ ]:
# 9. Predict
y_train_pred = xgb_model.predict(X_train_scaled)
y_test_pred  = xgb_model.predict(X_test_scaled)

# 10. Evaluate
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse  = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2   = r2_score(y_train, y_train_pred)
test_r2    = r2_score(y_test, y_test_pred)
mae        = mean_absolute_error(y_test, y_test_pred)

print(f'Training set RMSE: {train_rmse:.3f}')
print(f'Testing set RMSE: {test_rmse:.3f}')
print(f'Training set R²: {train_r2:.3f}')
print(f'Testing set R²: {test_r2:.3f}')
print(f'Testing set MAE: {mae:.3f}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))

# 1. Training set: hollow blue circles
ax.scatter(y_train, y_train_pred,
           facecolors='none',          # Hollow
           edgecolors='tab:blue',
           linewidth=1.2,
           alpha=0.8,
           label='Training set')

# 2. Test set: hollow orange circles
ax.scatter(y_test, y_test_pred,
           facecolors='none',
           edgecolors='tab:orange',
           linewidth=1.2,
           alpha=0.8,
           label='Test set')

# 3. Perfect fit line
lim = [y.min() - 0.5, y.max() + 0.5]
ax.plot(lim, lim, 'r--', lw=2, label='Perfect-fit curve')

# 4. Other styling
ax.set_xlabel('True Value', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Value', fontsize=12, fontweight='bold')
ax.set_title('XGB', fontsize=14, fontweight='bold')  # Only change title here
ax.grid(True, alpha=0.3)
ax.legend()

# # 5. Metric text (upper right corner)
# text_str = (f'Train RMSE = {train_rmse:.3f}\n'
#             f'Train R² = {train_r2:.3f}\n'
#             f'Test RMSE = {test_rmse:.3f}\n'
#             f'Test R² = {test_r2:.3f}')
# ax.text(0.05, 0.95, text_str, transform=ax.transAxes,
#         fontsize=11, verticalalignment='top',
#         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# 6. Equal aspect ratio axes
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

In [ ]:
import re
import shap

In [ ]:
# ---------- 1. Rename function ----------
def short_name(s):
    s = re.sub(r'^MagpieData\s+', '', s)
    parts = s.split()
    if len(parts) >= 2:
        stat_type, prop_name = parts[0], ' '.join(parts[1:])
        stat_abbr = {'maximum':'max','minimum':'min','avg_dev':'avg_dev',
                     'mean':'mean'}.get(stat_type, stat_type)
        clean_prop = prop_name.replace(' ', '').replace('_', '')
        prop_abbr = clean_prop[:2] + clean_prop[-1] if len(clean_prop) >= 3 else clean_prop
        return f"{stat_abbr}_{prop_abbr}"
    return (s[:10] + '…') if len(s) > 10 else s

In [ ]:
# ---------- 2. Select top 20 important features ----------
feat_imp = pd.Series(xgb_model.feature_importances_, index=X.columns)
top20_orig = feat_imp.sort_values(ascending=False).head(20).index.tolist()

# Construct original name -> short name mapping
name_map = {orig: short_name(orig) for orig in top20_orig}
name_map['k(W/mK)'] = r'$\kappa$'   # Also give kl a latex name

# ---------- 3. SHAP calculation (only 20 columns) ----------
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train_scaled)   # Full feature SHAP

# Get indices of these 20 columns
idx_20 = [X.columns.get_loc(c) for c in top20_orig]
shap_20 = shap_values[:, idx_20]
X_20    = X_train_scaled[:, idx_20]
X_20_df = pd.DataFrame(X_20, columns=[name_map[c] for c in top20_orig])

In [ ]:
# ---------- 4. SHAP Plot 1: beeswarm ----------
plt.figure()
shap.summary_plot(shap_20, X_20_df, show=False, max_display=20)
plt.title('SHAP Beeswarm — Top 20 Features')
plt.tight_layout()
plt.show()

In [ ]:
# ---------- 5. SHAP Plot 2: bar ----------
plt.figure()
shap.summary_plot(shap_20, X_20_df, plot_type="bar", show=False, max_display=20)
plt.title('SHAP Bar — Top 20 Features')
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Only keep top 10 ----------
top10_orig = feat_imp.sort_values(ascending=False).head(10).index.tolist()
name_map10 = {orig: short_name(orig) for orig in top10_orig}
name_map10['k(W/mK)'] = r'$\kappa$'

# Extract data and SHAP values for these 10 columns
idx10 = [X.columns.get_loc(c) for c in top10_orig]
shap10 = shap_values[:, idx10]
X10    = X_train_scaled[:, idx10]
X10_df = pd.DataFrame(X10, columns=[name_map10[c] for c in top10_orig])

In [ ]:
# ---------- SHAP Plot 1: beeswarm ----------
plt.figure(facecolor='white')
shap.summary_plot(shap10, X10_df, show=False, max_display=10)
ax = plt.gca()
# Remove internal grid lines
ax.grid(False)

# Set tick label font size
ax.tick_params(axis='both', labelsize=13)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.9)
    spine.set_color('black')

plt.title('SHAP Beeswarm — Top 10 Features')
plt.tight_layout()
# Specify background color when saving
plt.savefig('..\png\shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ---------- SHAP Plot 2: bar ----------
plt.figure()
shap.summary_plot(shap10, X10_df, plot_type="bar", show=False, max_display=10)
ax = plt.gca()
# Remove internal grid lines
ax.grid(False)

# Set tick label font size
ax.tick_params(axis='both', labelsize=13)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.9)
    spine.set_color('black')
plt.title('SHAP Bar — Top 10 Features')
plt.tight_layout()
plt.savefig('..\png\shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()

### Positive Impact Interval (Value Range)

In [ ]:
# ---------- 1. Data Preparation ----------
top10_orig = feat_imp.sort_values(ascending=False).head(8).index.tolist()
idx10 = [X.columns.get_loc(c) for c in top10_orig]
shap10 = shap_values[:, idx10]          # 10 columns SHAP
X10    = X_train_scaled[:, idx10]       # 10 columns original values
feature_names = [short_name(c) for c in top10_orig]

X10_df = pd.DataFrame(X10, columns=feature_names)
shap10_df = pd.DataFrame(shap10, columns=feature_names)

def optimized_negative_zone_near_best(feat_col, shap_col, range_percent=30):
    """
    Find optimal interval near negative SHAP values (more focused method)
    Steps:
    1. Find the point with most negative SHAP value (optimal value)
    2. Select an interval near the optimal value (e.g., optimal value ±30% range)
    3. Only consider samples with negative SHAP in this interval
    """
    tmp = pd.DataFrame({'x': feat_col, 'shap': shap_col})
    neg_samples = tmp.loc[tmp.shap < 0]

    if neg_samples.empty:
        return None, None

    # 1. Find optimal value (point with minimum SHAP value)
    best_idx = neg_samples['shap'].idxmin()
    best_val = neg_samples.loc[best_idx, 'x']

    # 2. Calculate feature value range
    feat_range = tmp['x'].max() - tmp['x'].min()

    # 3. Determine interval range (can adjust range_percent)
    range_width = (range_percent / 100) * feat_range
    left = max(tmp['x'].min(), best_val - range_width/2)
    right = min(tmp['x'].max(), best_val + range_width/2)

    # 4. Filter negative SHAP samples in this interval
    zone_samples = neg_samples[(neg_samples['x'] >= left) & (neg_samples['x'] <= right)]

    if zone_samples.empty:
        # If no negative SHAP samples in interval, return small range near optimal value
        left = best_val - 0.1 * feat_range
        right = best_val + 0.1 * feat_range
    else:
        # Use actual range of samples in the interval
        left = zone_samples['x'].min()
        right = zone_samples['x'].max()

    return pd.Interval(left, right, closed='both'), best_val

def optimized_negative_zone_quantile(feat_col, shap_col, quantile_range=(0.25, 0.75)):
    """
    Method 2: Quantile-based focused interval
    Only consider better performing part of negative SHAP values (e.g., middle 50%)
    """
    tmp = pd.DataFrame({'x': feat_col, 'shap': shap_col})
    neg_samples = tmp.loc[tmp.shap < 0]

    if neg_samples.empty:
        return None, None

    # Sort by SHAP value (from most negative to least negative)
    neg_samples_sorted = neg_samples.sort_values('shap')

    # Calculate quantile positions
    n_samples = len(neg_samples_sorted)
    lower_idx = int(quantile_range[0] * n_samples)
    upper_idx = int(quantile_range[1] * n_samples)

    # Take middle part (remove extreme values)
    middle_samples = neg_samples_sorted.iloc[lower_idx:upper_idx]

    # Optimal value: point with minimum SHAP value
    best_idx = neg_samples['shap'].idxmin()
    best_val = neg_samples.loc[best_idx, 'x']

    left = middle_samples['x'].min()
    right = middle_samples['x'].max()

    return pd.Interval(left, right, closed='both'), best_val

# Select which method to use (using method 2 here, more stable)
results = {}
for fn in feature_names:
    interval, best_val = optimized_negative_zone_quantile(
        X10_df[fn], shap10_df[fn], quantile_range=(0.25, 0.75)
    )
    results[fn] = {'interval': interval, 'best_val': best_val}

# ---------- 4. Visualization ----------
fig, ax = plt.subplots(2, 4, figsize=(18, 8))
axes = ax.ravel()

features_to_plot = min(8, len(feature_names))

for i in range(features_to_plot):
    fn = feature_names[i]
    ax_i = axes[i]

    # Scatter plot
    neg_mask = shap10_df[fn] < 0
    ax_i.scatter(X10_df[fn][neg_mask], shap10_df[fn][neg_mask],
                 c='tab:blue', s=15, alpha=0.6)
    ax_i.scatter(X10_df[fn][~neg_mask], shap10_df[fn][~neg_mask],
                 c='tab:red', s=15, alpha=0.6)

    # Mark optimal value and interval
    interval = results[fn]['interval']
    if interval:
        # Optimal value line
        ax_i.axvline(results[fn]['best_val'], color='k', ls='--',
                     linewidth=2, label=f'best = {results[fn]["best_val"]:.2f}')

    # Set chart properties
    ax_i.set_xlabel(f'{fn} value', fontsize=12)
    ax_i.set_ylabel('SHAP value', fontsize=12)

    # Tick settings
    ax_i.tick_params(
        axis='both',
        which='major',
        direction='out',
        length=3,
        width=1.0,
        color='black',
        bottom=True,
        left=True,
        top=False,
        right=False
    )

    ax_i.tick_params(axis='x', labelsize=12)
    ax_i.tick_params(axis='y', labelsize=12)

    # Spine settings
    for spine in ax_i.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1)

    ax_i.grid(False)

    # Simplified legend (avoid too many entries)
    ax_i.legend(fontsize=12, loc='best')  # Set legend position to best location

    # Add label in upper left corner
    ax_i.text(-0.16, 1, f'{chr(97+i)})', transform=ax_i.transAxes, fontsize=12, fontweight='bold', va='top', ha='left')


# Hide extra subplots
for i in range(features_to_plot, 8):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig('..\png\optimal_range.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ---------- 5. Table Print ----------
print('>>> Focused Negative Impact Intervals & Optimal Values (to minimize prediction) <<<')
print('=' * 60)
for fn, v in results.items():
    if v['interval'] is None:
        print(f"{fn:20s} : No obvious negative interval")
    else:
        left, right = v['interval'].left, v['interval'].right
        best_val = v['best_val']
        zone_width = right - left

        # Add some statistics
        zone_samples = X10_df[(X10_df[fn] >= left) & (X10_df[fn] <= right)]
        if len(zone_samples) > 0:
            coverage = len(zone_samples) / len(X10_df) * 100
            print(f"{fn:20s} : [{left:7.3f}, {right:7.3f}] (width: {zone_width:.3f})")
            print(f"{' ':20s}   → best = {best_val:7.3f} | coverage: {coverage:.1f}% of samples")
        else:
            print(f"{fn:20s} : [{left:7.3f}, {right:7.3f}]")
            print(f"{' ':20s}   → best = {best_val:7.3f}")

In [ ]:
# ---------- Create plot_df for plotting ----------
plot_data = []
for fn in feature_names:
    v = results[fn]
    if v['interval'] is not None:
        plot_data.append({
            'feat': fn,
            'left': v['interval'].left,
            'right': v['interval'].right,
            'best': v['best_val']
        })

# Create DataFrame
plot_df = pd.DataFrame(plot_data)

In [ ]:
# Sort by interval center position (optional)
plot_df = plot_df.sort_values('best', ascending=True).reset_index(drop=True)

# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(8, max(4, 0.35*len(plot_df))))

# Set white background
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

y_pos = np.arange(len(plot_df))

# Draw horizontal interval bars - using light blue
for i, row in plot_df.iterrows():
    ax.barh(y_pos[i], width=row.right-row.left,
            left=row.left, height=0.6,
            color='skyblue', edgecolor='k', linewidth=0.5)

# Best value vertical line (red dots)
ax.scatter(plot_df.best, y_pos, color='red', s=25, zorder=3, label='best value')

# Axis decoration
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df.feat, fontsize=12)
ax.set_xlabel('Standardized value', fontsize=12)
ax.legend()

# Hide grid lines
ax.grid(False)

# Set all four spines to black
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_color('black')

# Set tick style
ax.tick_params(
    axis='both',           # Set both x and y axes
    which='major',         # Major ticks
    direction='out',       # Ticks outward
    length=3,              # Tick length
    width=1.0,             # Tick width
    color='black',         # Tick color
    bottom=True,           # Show bottom ticks
    left=True,             # Show left ticks
    labelsize=13           # Tick label size
)

# Add title
#ax.set_title('Optimal Ranges & Best Values (for Minimizing Predictions)')

plt.tight_layout()
plt.savefig('..\png\optimal_ranges_horizontal.png', dpi=100, bbox_inches='tight')
plt.show()